In [ ]:
import time
start_time = time.time()

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import CrossEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix


In [ ]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Selected device: {device}")


In [ ]:
model_name = "cross-encoder/ms-marco-MiniLM-L-6-v2"
model = CrossEncoder(model_name, device=device)
print(f"Loaded model: {model_name}")


In [ ]:
dataset = load_dataset("glue", "mrpc", split="validation")
print(f"Validation examples: {len(dataset)}")

preview_df = dataset.select(range(min(5, len(dataset)))).to_pandas()
print(preview_df[["sentence1", "sentence2", "label"]].to_string(index=False))


In [ ]:
sentence_pairs = list(zip(dataset["sentence1"], dataset["sentence2"]))
true_labels = np.array(dataset["label"])

batch_size = 64
scores = np.asarray(model.predict(sentence_pairs, batch_size=batch_size, show_progress_bar=True), dtype=float)
predictions = (scores >= 0.5).astype(int)

print(f"Completed inference for {len(predictions)} examples.")
print(f"Score range: min={scores.min():.4f}, max={scores.max():.4f}")


In [ ]:
threshold = 0.5
accuracy = accuracy_score(true_labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(
    true_labels,
    predictions,
    average="binary",
    zero_division=0
)
cm = confusion_matrix(true_labels, predictions)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print("Confusion Matrix:")
print(cm)


In [ ]:
results_df = pd.DataFrame([
    {
        "model_name": model_name,
        "split": "validation",
        "num_examples": len(dataset),
        "threshold": threshold,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "device": device
    }
])

print(results_df.to_string(index=False))


In [ ]:
score_summary = pd.Series(scores).describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
score_summary_df = score_summary.rename("score").to_frame()
print("Score distribution summary:")
print(score_summary_df.to_string())

negative_rate = float((predictions == 0).mean())
positive_rate = float((predictions == 1).mean())
below_threshold = int((scores < threshold).sum())
above_or_equal_threshold = int((scores >= threshold).sum())

distribution_df = pd.DataFrame([
    {
        "threshold": threshold,
        "below_threshold": below_threshold,
        "above_or_equal_threshold": above_or_equal_threshold,
        "predicted_negative_rate": negative_rate,
        "predicted_positive_rate": positive_rate
    }
])
print("\nThreshold summary:")
print(distribution_df.to_string(index=False))


In [ ]:
examples_df = dataset.to_pandas()[["sentence1", "sentence2", "label"]].copy()
examples_df = examples_df.rename(columns={"label": "true_label"})
examples_df["score"] = scores
examples_df["predicted_label"] = predictions
examples_df["abs_distance_to_threshold"] = np.abs(examples_df["score"] - threshold)

print("First 10 scored examples:")
print(examples_df.head(10).to_string(index=False))

nearest_df = examples_df.sort_values("abs_distance_to_threshold", ascending=True).head(10)
print("\nNearest-to-threshold examples:")
print(nearest_df[["sentence1", "sentence2", "true_label", "score", "predicted_label", "abs_distance_to_threshold"]].to_string(index=False))

mismatches_df = examples_df[examples_df["true_label"] != examples_df["predicted_label"]].copy()
mismatches_df = mismatches_df.sort_values("abs_distance_to_threshold", ascending=True)
print(f"\nMismatches: {len(mismatches_df)}")
if len(mismatches_df) > 0:
    print(mismatches_df.head(10)[["sentence1", "sentence2", "true_label", "score", "predicted_label", "abs_distance_to_threshold"]].to_string(index=False))


In [ ]:
elapsed_seconds = time.time() - start_time
print(f"Total runtime (seconds): {elapsed_seconds:.2f}")
